# Buy / Not-Buy Label Analysis

This notebook analyses and validates parameters for the **binary buy/not-buy labeling** method.

Labels each tick as:
- **Buy (1)** — A long position entered here would hit the profit-take (PT) barrier *before* the stop-loss (SL) barrier or timeout.
- **Not-buy (0)** — SL hit first, or timeout elapsed with no barrier hit.

Barriers are dynamic, sized relative to the current bid-ask spread:
```
PT level = entry_mid × (1 + pt_multiplier × spread_pct_ema)
SL level = entry_mid × (1 − sl_multiplier × spread_pct_ema)
```

Workflow:
1. **Load Execution Price Data** — realistic bid/ask/mid prices for the chosen market.
2. **Calculate Spread & Volatility** — compute `spread_pct_ema`, `volatility_ema`, and per-tick barrier levels.
3. **Generate Labels** — run the numba-optimised forward scan.
4. **Filter by Vol/Spread Ratio** — keep only high-quality entry points.
5. **PnL Analysis** — validate label quality via realistic `buy_pnl`.
6. **Label Distribution Over Time** — check for temporal stability.
7. **Structural Profitability (Gambler's Ruin)** — theoretical buy-rate sanity check.
8. **Save Labels** — persist to Parquet store for ML training.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import logging
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from datetime import timedelta
from makalu import settings as sys_env
from makalu.storage.parquet_store import ParquetStore
from qate.util.dt_range import DtRange
from training import config
from labeling.buy_label import (
    BuyLabelConfig,
    calculate_spread_and_barriers,
    compute_buy_labels,
)
from labeling.triple_barrier import (
    ts_index,
    filter_by_vol_spread_ratio,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

SAVE_LABELS = True

(
    FEATURE_MARKET,
    LABEL_MARKET,
    EXECUTION_SIZE,
    TARGET_COLUMN,
    MIN_VOL_SPREAD_RATIO,
    BUY_CONFIG,
) = config.setup_buy_config()

print(f'BuyLabelConfig param_id: {BUY_CONFIG.param_id}')
print(f'Label market:   {LABEL_MARKET.id}')
print(f'Execution size: {EXECUTION_SIZE}')

DATA_START_DATE = '20260105'
DATA_END_DATE   = '20260114'
DT_RANGE = DtRange.from_strings(DATA_START_DATE, DATA_END_DATE)

## 1. Load Execution Price Data

In [ ]:
read_range = DtRange(
    DT_RANGE.start_dt - timedelta(hours=8),
    DT_RANGE.end_dt   + timedelta(hours=8),
)

print(f'Loading execution prices for {LABEL_MARKET.id} (size: {EXECUTION_SIZE})...')
price_store = ParquetStore(sys_env.get_parquet_root_dir()).create_execution_price(
    LABEL_MARKET, execution_size_str=EXECUTION_SIZE
)
df = price_store.read(read_range)
display(df.shape, df.head())

df = ts_index(df)
print(f'Tick count after ts_index: {len(df):,}')

## 2. Calculate Spread / Volatility & Visualize

Compute `spread_pct_ema`, `volatility_ema`, and the per-tick barrier fractions
`pt_pct` / `sl_pct` using `calculate_spread_and_barriers`.

Key difference from triple-barrier: **PT and SL are asymmetric** —
`pt_pct = pt_multiplier × spread_pct_ema` vs `sl_pct = sl_multiplier × spread_pct_ema`.

In [ ]:
df = calculate_spread_and_barriers(df, BUY_CONFIG)
print(f'Ticks after spread/barrier calc: {len(df):,}')
display(df[['mid', 'bid', 'ask', 'spread_pct_ema', 'volatility_ema', 'pt_pct', 'sl_pct']].describe().T)

In [ ]:
# 2.1 Spread vs Volatility over time
fig, ax = plt.subplots(figsize=(40, 12))

ax.plot(df.index, df['volatility_ema'] / df['spread_pct_ema'], color='tab:blue', linewidth=0.8, label='vol / spread')
for level in [1.0, 2.0, 3.0, 4.0]:
    ax.axhline(y=level, color='tab:red', linestyle='--', linewidth=0.8, alpha=0.7, label=f'y={level:.1f}')
ax.set_ylim(bottom=0, top=20)
ax.set_xlabel('Time')
ax.set_ylabel('vol / spread ratio')
ax.set_title(f'{LABEL_MARKET.symbol.name} — Volatility EMA / Spread EMA')
ax.legend()
plt.tight_layout()
plt.show()

ratio = df['volatility_ema'] / df['spread_pct_ema']
total = len(ratio)
print(f'Total ticks: {total:,}')
print(f'Mean spread_pct_ema:  {df["spread_pct_ema"].mean():.6f}  ({df["spread_pct_ema"].mean() * 10000:.4f} bps)')
print(f'Mean volatility_ema:  {df["volatility_ema"].mean():.6f}  ({df["volatility_ema"].mean() * 10000:.4f} bps)')
print()
print('vol/spread threshold  | count      | pct')
print('----------------------|------------|--------')
for level in [1.0, 2.0, 3.0, 4.0]:
    n = (ratio > level).sum()
    print(f'  > {level:.1f}               | {n:>10,} | {n / total:.1%}')

In [ ]:
# 2.2 Asymmetric barrier levels vs spread
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['pt_pct'] * 10000, color='green', linewidth=0.8, label=f'PT ({BUY_CONFIG.pt_multiplier}× spread)')
ax.plot(df.index, df['sl_pct'] * 10000, color='red',   linewidth=0.8, label=f'SL ({BUY_CONFIG.sl_multiplier}× spread)')
ax.set_xlabel('Time')
ax.set_ylabel('Barrier size (bps)')
ax.set_title('Dynamic PT vs SL Barrier Sizes Over Time')
ax.legend()
plt.tight_layout()
plt.show()

print(f'PT / SL ratio (avg): {(df["pt_pct"] / df["sl_pct"]).mean():.2f}x  '
      f'(target: {BUY_CONFIG.pt_multiplier / BUY_CONFIG.sl_multiplier:.2f}x)')

## 3. Generate Buy / Not-Buy Labels

The numba kernel scans forward from each tick:
- `mid >= PT level` → **Buy (1)**, exit at that tick
- `mid <= SL level` → **Not-buy (0)**, exit at that tick
- timeout elapsed → **Not-buy (0)**, exit at last in-window tick

`buy_pnl = bid[exit] − ask[entry]`

In [ ]:
print(f'Generating buy/not-buy labels for {len(df):,} ticks...')
result = compute_buy_labels(df, BUY_CONFIG)

labels_df = result.labels_df
print(f'Label count: {len(labels_df):,}')
print(f'  Buy (1):     {result.buy_count:,}  ({result.buy_rate:.1%})')
print(f'  Not-buy (0): {result.not_buy_count:,}  ({1 - result.buy_rate:.1%})')

# Join back to df for downstream analysis
df['label']   = labels_df['label']
df['buy_pnl'] = labels_df['buy_pnl']

display(labels_df.head())

## 4. Filter by Vol / Spread Ratio

Filter to keep only ticks where `volatility_ema > spread_pct_ema × MIN_VOL_SPREAD_RATIO`.
Labels are computed first (on all ticks) so the filter does **not** bias the label values.

In [ ]:
def plot_buy_labels(labels: pd.Series, config: BuyLabelConfig) -> None:
    label_counts = labels.value_counts().sort_index()
    label_map = {
        1: f'Buy     (PT = {config.pt_multiplier}× spread)',
        0: f'Not-buy (SL = {config.sl_multiplier}× spread | timeout {config.timeout_sec}s)',
    }
    label_counts.rename(index=label_map, inplace=True)

    total = label_counts.sum()
    print(f'Total samples: {total:,}')
    summary = pd.DataFrame({
        'count': label_counts,
        'pct':   (label_counts / total * 100).round(2),
    })
    print(summary)

    plt.figure(figsize=(8, 5))
    ax = sns.barplot(x=summary.index, y=summary['count'], palette=['red', 'green'])
    ax.set_title('Buy / Not-Buy Label Distribution')
    ax.set_ylabel('Count')
    ax.set_xlabel('Outcome')
    for p, pct in zip(ax.patches, summary['pct']):
        ax.text(p.get_x() + p.get_width() / 2, p.get_height() + 0.1,
                f'{pct}%', ha='center', va='bottom')
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.show()


print('=== Before vol/spread filter ===')
plot_buy_labels(df['label'], BUY_CONFIG)

filtered_df = filter_by_vol_spread_ratio(df, MIN_VOL_SPREAD_RATIO)

print(f'\n=== After vol/spread filter (ratio={MIN_VOL_SPREAD_RATIO}) ===')
plot_buy_labels(filtered_df['label'], BUY_CONFIG)

print(f'\nRemoved: {len(df) - len(filtered_df):,} rows '
      f'({(len(df) - len(filtered_df)) / len(df):.1%})')

# Use filtered data for downstream analysis
labels_df = filtered_df[['label', 'buy_pnl']].copy()

## 5. PnL Analysis

`buy_pnl = bid[exit] − ask[entry]` — a long position entered at the ask and exited at the bid.

For a **well-calibrated** label we expect:
- `buy_pnl` when `label=1` (Buy) should be **positive** on average.
- `buy_pnl` when `label=0` (Not-buy) should be **negative** on average.
- The average buy_pnl on Buy samples should comfortably exceed the spread.

In [ ]:
# Exclude edge effect (last timeout_sec — no full forward window)
cutoff = labels_df.index.max() - pd.Timedelta(seconds=BUY_CONFIG.timeout_sec)
pnl_df = labels_df.loc[labels_df.index <= cutoff].copy()

avg_mid = filtered_df['mid'].mean()
pnl_df['buy_pnl_bps'] = (pnl_df['buy_pnl'] / avg_mid) * 10_000

print(f'Samples for PnL analysis (after edge trim): {len(pnl_df):,}')
print()

for label_val, name, color in [(1, 'Buy (1)', 'green'), (0, 'Not-buy (0)', 'red')]:
    subset = pnl_df[pnl_df['label'] == label_val]
    if len(subset) == 0:
        continue
    bps = subset['buy_pnl_bps']
    print(f'{name}: {len(subset):,} samples')
    print(f'  avg buy_pnl = {subset["buy_pnl"].mean():.5f}  ({bps.mean():.2f} bps)')
    print(f'  total       = {subset["buy_pnl"].sum():.4f}  ({bps.sum():.1f} bps)')
    print(f'  win rate (pnl > 0): {(subset["buy_pnl"] > 0).mean():.1%}')
    print()

print('Overall (all samples):')
print(f'  avg buy_pnl = {pnl_df["buy_pnl"].mean():.5f}  ({pnl_df["buy_pnl_bps"].mean():.2f} bps)')
print(f'  win rate    = {(pnl_df["buy_pnl"] > 0).mean():.1%}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

buy_df     = pnl_df[pnl_df['label'] == 1]
not_buy_df = pnl_df[pnl_df['label'] == 0]

# 1. PnL distribution by label
ax = axes[0, 0]
ax.hist(buy_df['buy_pnl_bps'],     bins=60, alpha=0.6, color='green',
        label=f'Buy (n={len(buy_df):,})')
ax.hist(not_buy_df['buy_pnl_bps'], bins=60, alpha=0.6, color='red',
        label=f'Not-buy (n={len(not_buy_df):,})')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('buy_pnl (bps)')
ax.set_ylabel('Count')
ax.set_title('Buy PnL Distribution by Label')
ax.legend()

# 2. Average PnL by label (bar)
ax = axes[0, 1]
avgs  = [buy_df['buy_pnl_bps'].mean(), not_buy_df['buy_pnl_bps'].mean()]
bars = ax.bar(['Buy (1)', 'Not-buy (0)'], avgs, color=['green', 'red'], alpha=0.8, edgecolor='black')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('Average buy_pnl (bps)')
ax.set_title('Average buy_pnl by Label')
for bar, v in zip(bars, avgs):
    ax.text(bar.get_x() + bar.get_width() / 2, v + (0.02 if v >= 0 else -0.05),
            f'{v:.2f}', ha='center', va='bottom' if v >= 0 else 'top', fontsize=10)

# 3. Cumulative buy_pnl over time
ax = axes[1, 0]
sorted_pnl = pnl_df.sort_index()
ax.plot(sorted_pnl.index, sorted_pnl['buy_pnl'].cumsum(), color='blue', linewidth=1, label='Always-long')
buy_only = sorted_pnl[sorted_pnl['label'] == 1]['buy_pnl']
buy_cum  = buy_only.reindex(sorted_pnl.index, fill_value=0).cumsum()
ax.plot(sorted_pnl.index, buy_cum, color='green', linewidth=1, label='Buy-only (label=1)')
ax.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Time')
ax.set_ylabel('Cumulative PnL')
ax.set_title('Cumulative buy_pnl Over Time')
ax.legend()
ax.tick_params(axis='x', rotation=30)

# 4. Rolling buy rate
ax = axes[1, 1]
window = 500
rolling_buy_rate = sorted_pnl['label'].rolling(window, min_periods=100).mean()
ax.plot(sorted_pnl.index, rolling_buy_rate, color='purple', linewidth=1)
ax.axhline(y=result.buy_rate, color='green', linestyle='--', linewidth=1,
           label=f'Overall buy rate ({result.buy_rate:.1%})')
ax.set_xlabel('Time')
ax.set_ylabel('Rolling Buy Rate')
ax.set_title(f'Rolling Buy Rate (window={window})')
ax.set_ylim(0, 1)
ax.legend()
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print('\nDetailed buy_pnl statistics by label (bps):')
display(pnl_df.groupby('label')['buy_pnl_bps'].describe().round(3))

## 6. Label Distribution Over Time

Check that the buy rate is stable across the data period — a severely non-stationary buy rate
suggests the chosen parameters may overfit to a particular market regime.

In [ ]:
# Hourly buy rate
hourly = filtered_df['label'].resample('1h').agg(['sum', 'count'])
hourly.columns = ['buy_count', 'total']
hourly['buy_rate'] = hourly['buy_count'] / hourly['total'].clip(lower=1)

fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharex=True)

axes[0].bar(hourly.index, hourly['buy_rate'], width=0.03, color='green', alpha=0.7)
axes[0].axhline(y=result.buy_rate, color='black', linestyle='--',
                label=f'Overall ({result.buy_rate:.1%})')
axes[0].set_ylabel('Buy Rate')
axes[0].set_title('Hourly Buy Rate')
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].bar(hourly.index, hourly['total'], width=0.03, color='steelblue', alpha=0.7)
axes[1].set_ylabel('Tick Count')
axes[1].set_title('Hourly Tick Count (after vol/spread filter)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(f'Buy rate — mean: {hourly["buy_rate"].mean():.1%}, '
      f'std: {hourly["buy_rate"].std():.1%}, '
      f'min: {hourly["buy_rate"].min():.1%}, '
      f'max: {hourly["buy_rate"].max():.1%}')

## 7. Structural Profitability — Gambler's Ruin

From the Gambler's Ruin analogy, the theoretical probability that a long trade hits the **SL before PT** is:

```
P(Loss) ≈ 0.5 + 0.5 × (spread / volatility)
```

With **asymmetric barriers** the formula adjusts: the effective 'win distance' is larger (PT further away),
so the theoretical P(Buy) also adjusts:

```
P(Buy)  ≈  sl_multiplier / (pt_multiplier + sl_multiplier)
         × (1 − spread/volatility)     [rough first-order approximation]
```

Use this as a **sanity check**: if the observed buy rate closely matches the Gambler's Ruin
prediction, the labels are behaving as expected.

In [ ]:
MAX_THEORETICAL_LOSS_RATE = 0.65   # alert threshold

s = filtered_df['spread_pct_ema']
v = filtered_df['volatility_ema']

s_div_v = (s / v).clip(upper=5.0)
filtered_df['theoretical_loss_p'] = 0.5 + 0.5 * s_div_v
filtered_df['theoretical_buy_p']  = 1.0 - filtered_df['theoretical_loss_p']

avg_loss_p = filtered_df['theoretical_loss_p'].mean()
avg_buy_p  = filtered_df['theoretical_buy_p'].mean()

print('--- Structural Profitability Analysis (Gambler\'s Ruin) ---')
print(f'Avg spread_pct_ema:         {s.mean():.6f}')
print(f'Avg volatility_ema:         {v.mean():.6f}')
print(f'Avg s/v ratio:              {s_div_v.mean():.3f}')
print()
print(f'Theoretical P(Loss):        {avg_loss_p:.2%}')
print(f'Theoretical P(Buy):         {avg_buy_p:.2%}')
print(f'Observed   buy rate:        {result.buy_rate:.2%}  (after vol/spread filter: {filtered_df["label"].mean():.2%})')
print()
if avg_loss_p > MAX_THEORETICAL_LOSS_RATE:
    print(f'WARNING: avg theoretical loss rate ({avg_loss_p:.1%}) exceeds threshold ({MAX_THEORETICAL_LOSS_RATE:.1%}).')
    print('Consider tightening the vol/spread filter or adjusting barrier multipliers.')
else:
    print(f'OK: avg theoretical loss rate ({avg_loss_p:.1%}) is within threshold ({MAX_THEORETICAL_LOSS_RATE:.1%}).')

In [ ]:
# Plot theoretical buy probability distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(filtered_df['theoretical_buy_p'], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax.axvline(x=avg_buy_p, color='green', linestyle='--', label=f'Mean = {avg_buy_p:.1%}')
ax.axvline(x=result.buy_rate, color='orange', linestyle='--',
           label=f'Observed buy rate = {result.buy_rate:.1%}')
ax.set_xlabel('Theoretical P(Buy)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Theoretical P(Buy)')
ax.legend()

ax = axes[1]
scatter_sample = filtered_df[['spread_pct_ema', 'volatility_ema', 'label']].sample(
    min(5000, len(filtered_df)), random_state=42
)
colors = scatter_sample['label'].map({1: 'green', 0: 'red'})
ax.scatter(scatter_sample['spread_pct_ema'] * 10_000,
           scatter_sample['volatility_ema']  * 10_000,
           c=colors, alpha=0.3, s=5)
ax.set_xlabel('spread_pct_ema (bps)')
ax.set_ylabel('volatility_ema (bps)')
ax.set_title('Label vs Spread / Volatility\n(green=Buy, red=Not-buy, sample of 5k)')

plt.tight_layout()
plt.show()

## 8. Save Labels to Parquet

In [ ]:
if SAVE_LABELS:
    parquet_store = ParquetStore(sys_env.get_parquet_root_dir()).create_label(
        'BuyLabel_tmp',
        LABEL_MARKET,
        BUY_CONFIG.param_id,
    )
    parquet_store.add_dataframe(filtered_df, ts_column=None)
    parquet_store.close()
    print(f'Saved {len(filtered_df):,} labelled ticks to Parquet (param_id={BUY_CONFIG.param_id})')
    print(f'  Buy (1):     {int(filtered_df["label"].sum()):,}')
    print(f'  Not-buy (0): {int((filtered_df["label"] == 0).sum()):,}')
else:
    print('SAVE_LABELS=False — skipping write.')